To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Introducing FP8 precision training for faster RL inference. [Read Blog](https://docs.unsloth.ai/new/fp8-reinforcement-learning).

Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
except: get_numpy = "numpy"; get_pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
!uv pip install -qqq --upgrade     unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
!uv pip install -qqq {get_triton}
!uv pip install "huggingface_hub>=0.34.0" "datasets==4.3.0
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

We're also introducing how you can do `GSPO` inside of Unsloth as well!

The goal of this notebook is to make a vision language model solve maths problems via reinforcement learning given an image input like below:

<img src="https://raw.githubusercontent.com/lupantech/MathVista/main/assets/our_new_3_datasets.png" alt="Alt text" height="256">

In [2]:
from unsloth import FastVisionModel
import torch
max_seq_length = 16384 # Must be this long for VLMs
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-VL-7B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    gpu_memory_utilization = 0.8, # Reduce if out of memory
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-11-30 07:42:52.192959: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764488572.598456    1024 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764488572.710045    1024 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 11-30 07:43:15 [__init__.py:244] Automatically detected platform cuda.
ERROR 11-30 07:43:17 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.4: Fast Qwen2_5_Vl patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
INFO 11-30 07:43:39 [vllm_utils.py:702] Unsloth: Patching vLLM v1 graph capture
INFO 11-30 07:43:39 [vllm_utils.py:732] Unsloth: Patching vLLM v0 graph capture
Unsloth: Vision model detected, setting approx_max_num_seqs to 1
Unsloth: vLLM loa

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 11-30 07:43:55 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 11-30 07:43:55 [config.py:1472] Using max model len 16384
WARNING 11-30 07:43:55 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
WARNING 11-30 07:43:55 [arg_utils.py:1556] --enable-prefix-caching is not supported for multimodal models in V0 and has been disabled.
INFO 11-30 07:43:57 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=16384.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['embed_tokens', 'embedding', 'lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'router', 'visual', 'model.visual.blocks.31.mlp', 'model.visual.

...

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 11-30 07:44:34 [punica_selector.py:19] Using PunicaWrapperGPU.
INFO 11-30 07:44:35 [model_runner.py:1203] Model loading took 6.7940 GiB and 34.794715 seconds
INFO 11-30 07:45:24 [worker.py:294] Memory profiling takes 48.63 seconds
INFO 11-30 07:45:24 [worker.py:294] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.77) = 11.34GiB
INFO 11-30 07:45:24 [worker.py:294] model weights take 6.79GiB; non_torch_memory takes 0.03GiB; PyTorch activation peak memory takes 2.71GiB; the rest of the memory reserved for KV Cache is 1.81GiB.
INFO 11-30 07:45:25 [executor_base.py:113] # cuda blocks: 2118, # CPU blocks: 4681
INFO 11-30 07:45:25 [executor_base.py:118] Maximum concurrency for 16384 tokens per request: 2.07x
INFO 11-30 07:45:29 [vllm_utils.py:737] Unsloth: Running patched vLLM v0 `capture_model`.
INFO 11-30 07:45:29 [model_runner.py:1513] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To ru

Capturing CUDA graph shapes:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 11-30 07:45:30 [model_runner.py:1671] Graph capturing finished in 1 secs, took 0.04 GiB
INFO 11-30 07:45:30 [vllm_utils.py:744] Unsloth: Patched vLLM v0 graph capture finished in 1 secs.
INFO 11-30 07:45:31 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 55.64 seconds
Unsloth: Just some info: will skip parsing ['k_norm', 'post_attention_layernorm', 'q_norm', 'post_layernorm', 'layer_norm1', 'layer_norm2', 'attention_norm', 'ffn_norm', 'norm1', 'pre_feedforward_layernorm', 'input_layernorm', 'norm2', 'post_feedforward_layernorm', 'norm']
Performing substitution for additional_keys={'model.visual.patch_embed.proj.weight', 'model.visual.merger.ln_q.weight'}
Unsloth: Just some info: will skip parsing ['k_norm', 'post_attention_layernorm', 'q_norm', 'post_layernorm', 'layer_norm1', 'layer_norm2', 'cross_attn_post_attention_layernorm', 'attention_norm', 'ffn_norm', 'pre_feedforward_layernorm', 'input_layernorm', 'post_feedforward_layernorm', 'norm', 'cross_

In Unsloth, we share vLLM's weights directly, reducing VRAM usage by > 50%. vLLM also does not yet support LoRA on the vision layers, so we can only add them on the language layers. Vision GRPO still works though!

In [3]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True,  # False if not finetuning language layers
    finetune_attention_modules = True,  # False if not finetuning attention layers
    finetune_mlp_modules       = True,  # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [4]:
from datasets import Dataset
from PIL import Image
import pandas as pd
import os

# Paths from your description
CSV_PATH = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Train.csv"
IMG_DIR = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Image"

# 1. Load first 500 rows
train_df = pd.read_csv(CSV_PATH).head(500)

TEXT_CONTENT = """You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations"""

# 2. Create a HF Dataset with raw columns first
#    We keep Image_name and Label, we'll add decoded_image
raw_ds = Dataset.from_pandas(train_df, preserve_index=False)

# 3. Load, resize to 512x512, convert to RGB -> decoded_image
def load_and_resize(example):
    img_path = os.path.join(IMG_DIR, example["Image_name"])
    img = Image.open(img_path)
    img = img.resize((512, 512))
    if img.mode != "RGB":
        img = img.convert("RGB")
    example["decoded_image"] = img
    return example

dataset = raw_ds.map(load_and_resize)

# 4. Build MathVista-style prompt structure: list with image placeholder + text
def make_conversation(example):
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image"},  # placeholder; actual image kept in decoded_image
                {"type": "text", "text": TEXT_CONTENT},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": example["Label"]},
            ],
        },
    ]
    return {
        "prompt": prompt,
        "image": example["decoded_image"],  # keep image separate (like MathVista)
        "answer": example["Label"],
    }

dataset = dataset.map(make_conversation)

# 5. Remove old image column (if any) and rename decoded_image -> image
#    In our case, we already use 'image' key above, so we can just drop decoded_image.
if "decoded_image" in dataset.column_names:
    dataset = dataset.remove_columns("decoded_image")

# 6. Apply chat template to build final text prompt string
#    tokenizer must already be created (from your FastVisionModel)
def apply_template(example):
    example["prompt"] = tokenizer.apply_chat_template(
        example["prompt"],
        tokenize=False,
        add_generation_prompt=True,  # include assistant tag
    )
    return example

dataset = dataset.map(apply_template)

print(dataset)
print("Columns:", dataset.column_names)
print("Sample prompt:", dataset[0]["prompt"][:400])
print("Image type:", type(dataset[0]["image"]))


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['Image_name', 'Label', 'prompt', 'image', 'answer'],
    num_rows: 500
})
Columns: ['Image_name', 'Label', 'prompt', 'image', 'answer']
Sample prompt: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentionin
Image type: <class 'PIL.PngImagePlugin.PngImageFile'>


Here is the first example prompt in the dataset

In [5]:
dataset[0]["prompt"]

'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:\nPolitical: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.\nNonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.\nOutput a single word without providing any reasoning or explanations<|im_end|>\n<|im_start|>assistant\nPolitical<|im_end|>\n<|im_start|>assistant\n'

<a name="Inference"></a>
### Inference
Now let's try the model on the hundredth sample of the train dataset without training.


In [7]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)

outputs = model.fast_generate(
    {
        "prompt": dataset[100]["prompt"],
        "multi_modal_data": {"image": dataset[100]["image"]}
    },
    sampling_params,
)
print(outputs[0].outputs[0].text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

NonPolitical


<a name="Train"></a>
### Train the model

Now set up the `GRPO` Trainer and all configurations! Note we actually enable `GSPO` as well!

In [9]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    log_completions = False,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = 1024,
    max_completion_length = 1024,
    num_train_epochs = 0.5, # Set to 1 for a full training run
    # max_steps = 60,
    save_steps = 60,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # Below enables GSPO:
    importance_sampling_level = "sequence",
    mask_truncated_completions = False,
    loss_type = "dr_grpo",
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

During inference, you might encounter `addCriterion` or some weird gibberish outputs. Please read our [blog post](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl#qwen-2.5-vl-vision-rl-issues-and-quirks) on why this occurs. It seems to be an inherent thing inside of the model, and we can ignore this.

In [11]:
import re

def meme_correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    prompts: list[str]  – full text prompts (one per sample)
    completions: list[str] – model outputs (one per sample)
    answer: list[str] – ground truth labels, e.g. ["Political", "NonPolitical", ...]
    Returns: list[float] – reward per sample
    """

    rewards = []
    for prompt, completion, gold in zip(prompts, completions, answer):
        # Clean up completion
        text = completion.strip()
        # Remove any XML-ish or think tags if present
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        text = re.sub(r"<.*?>", "", text).strip()

        # Normalize
        pred = text.lower()
        gold_norm = gold.lower().strip()

        # Map any variants back to the two allowed labels
        if "nonpolitical" in pred or "non-political" in pred or "non political" in pred:
            pred_label = "nonpolitical"
        elif "political" in pred:
            pred_label = "political"
        else:
            pred_label = ""  # unknown / off-distribution

        gold_label = "nonpolitical" if "non" in gold_norm else "political"

        reward = 2.0 if pred_label == gold_label else 0.0
        rewards.append(reward)

    return rewards


In [12]:
trainer = GRPOTrainer(
    model = model,
    args = training_args,
    # Pass the processor to handle multimodal inputs
    processing_class = tokenizer,
    reward_funcs = [
        meme_correctness_reward_func
    ],
    train_dataset = dataset,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 8,332,536,832 (0.48% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / meme_correctness_reward_func / mean,rewards / meme_correctness_reward_func / std
1,0.000000,0.750000,0.500000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000000,0.750000,1.035098
2,-0.000200,1.500000,1.000000,2.875000,2.000000,3.000000,0.000000,2.875000,2.000000,3.000000,0.000010,1.500000,0.925820
3,-0.000200,1.500000,1.000000,2.875000,2.000000,3.000000,0.000000,2.875000,2.000000,3.000000,0.000008,1.500000,0.925820
4,0.000400,1.500000,0.577350,3.000000,2.000000,5.000000,0.000000,3.000000,2.000000,5.000000,0.000004,1.500000,0.925820
5,0.000000,1.750000,0.500000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000017,1.750000,0.707107
6,0.000000,1.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000005,1.000000,1.069045
7,0.000000,0.000000,0.000000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000018,0.000000,0.000000
8,-0.000300,1.000000,1.154701,2.625000,2.000000,3.000000,0.000000,2.625000,2.000000,3.000000,0.000105,1.000000,1.069045
9,0.000000,1.750000,0.500000,3.000000,3.000000,3.000000,0.000000,3.000000,3.000000,3.000000,0.000108,1.750000,0.707107
10,-0.000200,1.250000,1.077350,2.750000,2.000000,3.000000,0.000000,2.750000,2.000000,3.000000,0.000246,1.250000,1.035098


TrainOutput(global_step=125, training_loss=2.4039018144476198e-05, metrics={'train_runtime': 3119.8936, 'train_samples_per_second': 0.08, 'train_steps_per_second': 0.04, 'total_flos': 0.0, 'train_loss': 2.4039018144476198e-05})

<a name="Inference"></a>
### Inference



And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [13]:
model.save_lora("grpo_lora")

We try calling vLLM with our trained RL model:

In [14]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)

outputs = model.fast_generate(
    {
        "prompt": dataset[165]["prompt"],
        "multi_modal_data": {"image": dataset[165]["image"]}
    },
    sampling_params,
    lora_request = model.load_lora("grpo_lora"))
print(outputs[0].outputs[0].text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

NonPolitical


Verify LoRA is actually trained!

In [16]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [19]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

In [17]:
model.push_to_hub_merged("arafid/finetuned-500", tokenizer, save_method = "merged_16bit", token = "")

config.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [04:58<01:40, 100.61s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [05:34<00:00, 83.61s/it] 


Unsloth: Merge process complete. Saved to `/kaggle/working/arafid/finetuned-500`


Special Credits to [GAD-Cell](https://github.com/GAD-cell) for helping Unsloth create this notebook and bringing VLM GRPO into Unsloth!

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
